# Free-Energy Landscapes, in depth

A Free-Energy Landscape (FEL) turns *how often* a simulation visits a region of
conformational space into *how energetically favourable* that region is. This
notebook is a deep dive into QuickPCA's `compute_fel`:

- the **Boltzmann inversion** that defines the energy,
- the role of **temperature**,
- how **binning** (`n_bins`) and **smoothing** (`sigma`) shape the result,
- and how to **read minima and basins**, with custom matplotlib plots.

> QuickPCA was developed by **Gleb Novikov**. MIT licensed.

## The physics: Boltzmann inversion

At thermal equilibrium the probability of finding the system in a state is given
by the **Boltzmann distribution**, $P \propto e^{-F / k_\mathrm{B}T}$. Inverting
that relation lets us recover a *relative* free energy from a sampled probability
density $\rho$:

$$ F = -k_\mathrm{B}T \,\ln \rho $$

QuickPCA applies this over the 2-D space of the first two principal components:

1. Histogram the PC1/PC2 projections into a 2-D density $\rho$ (`n_bins` per axis).
2. Smooth $\rho$ with a Gaussian filter of width `sigma` (in bins) to suppress
   histogram noise.
3. Take $F = -k_\mathrm{B}T \ln \rho$, with $k_\mathrm{B}T$ set by `temperature`.
4. Shift so the global minimum is exactly $F = 0$ (energies are **relative**).

Empty bins (never visited) have $\rho = 0$ and are left as `NaN` — they are
regions the simulation never explored, not zero-energy regions.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np

from quickpca import compute_fel, compute_pca, load_trajectory

# A slightly finer stride than the quickstart gives a smoother landscape while
# staying fast.
traj = load_trajectory(
    "../data/reference.pdb",
    "../data/trajectory.nc",
    selection="name CA",
    interval=5,
)
pca = compute_pca(traj.coords, n_components=10, backend="numpy")
print(f"{traj.n_frames} frames projected onto "
      f"PC1 ({pca.explained_variance_ratio[0] * 100:.1f}%) and "
      f"PC2 ({pca.explained_variance_ratio[1] * 100:.1f}%)")

## A first landscape

We compute a baseline FEL at body temperature with moderate resolution and
smoothing, then plot it with our own matplotlib code so we control every visual
choice. The bundled `plot_report` produces a polished multi-panel figure; here we
build a focused, single-panel FEL to dissect it.

In [ ]:
fel = compute_fel(pca.projections, temperature=310.0, n_bins=50, sigma=1.0)

print(f"temperature : {fel.temperature:.0f} K")
print(f"kBT         : {fel.kBT:.3f} kJ/mol")
print(f"F grid      : {fel.F.shape}")
print(f"F max       : {np.nanmax(fel.F):.2f} kJ/mol")
print(f"visited bins: {np.isfinite(fel.F).sum()} / {fel.F.size}")

In [ ]:
import matplotlib.pyplot as plt


def plot_fel(fel, ax=None, title="Free-Energy Landscape", levels=30, show_path=True):
    "Custom single-panel FEL contour plot from a QuickPCA FELResult."
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))

    F = fel.F
    # Replace NaN (unvisited) with the max so contouring fills cleanly.
    F_plot = np.where(np.isnan(F), np.nanmax(F), F)
    XX, YY = np.meshgrid(fel.xcenters, fel.ycenters)
    lv = np.linspace(0, np.nanpercentile(F, 98), levels)

    cf = ax.contourf(XX, YY, F_plot.T, levels=lv, cmap="nipy_spectral", extend="max")
    ax.contour(XX, YY, F_plot.T, levels=lv[::5], colors="k", linewidths=0.3, alpha=0.4)
    cbar = plt.gcf().colorbar(cf, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(r"$F$  (kJ mol$^{-1}$)")

    if show_path:
        ax.plot(fel.pc1, fel.pc2, color="white", lw=0.4, alpha=0.4)
        ax.scatter(fel.pc1[0], fel.pc2[0], c="lime", s=90, marker="*",
                   edgecolors="k", lw=0.6, zorder=5, label="start")
        ax.scatter(fel.pc1[-1], fel.pc2[-1], c="red", s=90, marker="*",
                   edgecolors="k", lw=0.6, zorder=5, label="end")
        ax.legend(loc="upper right", fontsize=8)

    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"{title}  (T = {fel.temperature:.0f} K)")
    return ax


plot_fel(fel)
plt.tight_layout()
plt.show()

## Finding minima and basins

A **minimum** is a grid cell whose free energy is lower than all of its
neighbours; a **basin** is the low-energy region around such a minimum. Below we
locate local minima with a small neighbourhood scan, keeping only sufficiently
deep ones (within a `depth_cutoff` of the global minimum) so that shallow
histogram ripples don't masquerade as states.

We then annotate them on the landscape. The **deepest basin is the most populated
conformational state**; secondary basins are alternative states the molecule
visits, and the warmer ridges between them are the transition barriers.

A word of caution: with limited (strided) sampling and fine binning, a naive
local-minimum scan reports *many* shallow dips that are really histogram noise.
We use a strict depth cutoff (within ~0.5 kBT of the global minimum) to keep only
the genuinely deep wells, and even then the count should be read as indicative,
not definitive — better sampling and coarser binning give cleaner basin counts.

In [ ]:
def find_minima(fel, depth_cutoff=None):
    # Return (pc1, pc2, F) of local minima of the landscape. A cell is a local
    # minimum if it is finite and <= all 8 neighbours. If depth_cutoff is given,
    # only minima with F <= depth_cutoff are kept.
    F = fel.F
    nx, ny = F.shape
    minima = []
    for i in range(nx):
        for j in range(ny):
            v = F[i, j]
            if not np.isfinite(v):
                continue
            lo = i - 1 if i > 0 else 0
            hi = i + 2 if i < nx - 1 else nx
            lo2 = j - 1 if j > 0 else 0
            hi2 = j + 2 if j < ny - 1 else ny
            neigh = F[lo:hi, lo2:hi2]
            if v <= np.nanmin(neigh) + 1e-9:
                if depth_cutoff is None or v <= depth_cutoff:
                    minima.append((fel.xcenters[i], fel.ycenters[j], v))
    minima.sort(key=lambda t: t[2])
    return minima


# Keep only deep wells: within ~0.5 kBT of the global minimum.
cutoff = 0.5 * fel.kBT
minima = find_minima(fel, depth_cutoff=cutoff)
print(f"found {len(minima)} deep minima (within {cutoff:.2f} kJ/mol "
      f"= 0.5 kBT of the global minimum):")
for k, (x, y, f) in enumerate(minima, 1):
    tag = "  <- global minimum" if k == 1 else ""
    print(f"  #{k}: PC1={x:7.2f}  PC2={y:7.2f}  F={f:5.2f} kJ/mol{tag}")

In [ ]:
ax = plot_fel(fel, title="FEL with annotated basins")
for k, (x, y, f) in enumerate(minima, 1):
    ax.scatter(x, y, marker="o", s=160, facecolors="none",
               edgecolors="white", linewidths=1.8, zorder=6)
    ax.annotate(f"min {k}\n{f:.1f}", (x, y),
                color="white", fontsize=8, fontweight="bold",
                ha="center", va="center", zorder=7)
plt.tight_layout()
plt.show()

## How `temperature` rescales the landscape

Temperature enters only through $k_\mathrm{B}T$, a multiplicative prefactor. Raising
$T$ stretches the *vertical* energy scale (barriers and well depths grow in
kJ/mol) but leaves the *shape* — where the minima are — unchanged. Physically,
$F$ at a higher $T$ reports the energy needed to overcome barriers relative to a
larger thermal budget. The cell below shows the same landscape at three
temperatures: identical topography, rescaled colour bar.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, T in zip(axes, (200.0, 310.0, 400.0)):
    fel_T = compute_fel(pca.projections, temperature=T, n_bins=50, sigma=1.0)
    plot_fel(fel_T, ax=ax, title=f"T = {T:.0f} K", show_path=False)
    ax.set_title(f"T = {T:.0f} K  (max F = {np.nanmax(fel_T.F):.1f} kJ/mol)")
plt.tight_layout()
plt.show()

## How `n_bins` and `sigma` trade resolution against noise

These two parameters control the **bias–variance trade-off** of the density
estimate:

- **`n_bins`** sets the grid resolution. *Too few* bins blur distinct basins
  together (over-smoothing); *too many* bins leave most cells empty or with one or
  two samples, producing a spiky, noisy landscape — especially with limited
  sampling like our strided demo.
- **`sigma`** is the Gaussian smoothing width *in bins*. Larger `sigma` suppresses
  histogram noise and merges ripples, but excessive smoothing washes out real
  features and shifts apparent minima.

A practical recipe: pick `n_bins` so that well-populated basins span several cells,
then use the smallest `sigma` that removes obvious salt-and-pepper noise. The grid
below sweeps both.

In [ ]:
bins_list = (25, 50, 100)
sigma_list = (0.5, 1.0, 2.0)

fig, axes = plt.subplots(
    len(sigma_list), len(bins_list),
    figsize=(5 * len(bins_list), 4.2 * len(sigma_list)),
)
for r, sig in enumerate(sigma_list):
    for c, nb_ in enumerate(bins_list):
        fel_g = compute_fel(pca.projections, temperature=310.0, n_bins=nb_, sigma=sig)
        plot_fel(fel_g, ax=axes[r, c],
                 title=f"n_bins={nb_}, sigma={sig}", show_path=False)
fig.suptitle("FEL sensitivity to binning (columns) and smoothing (rows)",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Reading the grid: along the top row (`sigma=0.5`) fine binning (`n_bins=100`)
looks grainy because each cell holds few frames. Moving down (more smoothing) and
left (coarser bins) progressively merges features. The middle cell
(`n_bins=50, sigma=1.0`) is the balanced default — enough resolution to separate
basins, enough smoothing to suppress sampling noise.

## A 1-D free-energy profile along PC1

Projecting the landscape onto a single principal component gives a **1-D
free-energy profile** — often the clearest way to count states and read off
barrier heights. We Boltzmann-invert the 1-D histogram of the PC1 projections
directly (the same $-k_\mathrm{B}T\ln\rho$ recipe), so dips are stable states and
bumps are barriers between them.

In [ ]:
def free_energy_1d(values, kBT, n_bins=60, sigma=1.0):
    from scipy.ndimage import gaussian_filter1d

    hist, edges = np.histogram(values, bins=n_bins, density=True)
    hist = gaussian_filter1d(hist, sigma=sigma)
    with np.errstate(divide="ignore"):
        F = np.where(hist > 0, -kBT * np.log(hist), np.nan)
    F -= np.nanmin(F)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, F


c1, F1 = free_energy_1d(fel.pc1, fel.kBT)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(c1, F1, color="navy", lw=2)
ax.fill_between(c1, F1, np.nanmax(F1), color="navy", alpha=0.08)
ax.set_xlabel("PC1")
ax.set_ylabel(r"$F$  (kJ mol$^{-1}$)")
ax.set_title(f"1-D free-energy profile along PC1  (T = {fel.temperature:.0f} K)")
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

depth = np.nanmax(F1)
print(f"PC1 profile spans {depth:.1f} kJ/mol "
      f"(~{depth / fel.kBT:.1f} kBT) from its deepest well to the highest sampled barrier.")

## Takeaways

- A FEL is a **Boltzmann inversion** of sampled density: $F = -k_\mathrm{B}T\ln\rho$,
  shifted so the global minimum is 0.
- **`temperature`** rescales the energy axis without moving the minima.
- **`n_bins`** and **`sigma`** trade resolution against noise — the
  `n_bins=50, sigma=1.0` default is a sensible starting point.
- **Minima are conformational states; basins are how populated they are; ridges
  are the barriers between them.** A 1-D profile along a PC is often the cleanest
  way to count states and read barrier heights.
- Always remember these energies are **relative** and **only as trustworthy as the
  sampling** — unvisited regions are `NaN`, not low-energy.

---
*QuickPCA — developed by Gleb Novikov. MIT licensed.*